<a href="https://colab.research.google.com/github/Rohil121/bharat-portfolio-lab/blob/v0.6-ml-trading-research/notebooks/06_ml_trading_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!printf "Y\n\n" | env BROWSER=echo stdbuf -oL -eL gh auth login \
  --hostname github.com \
  --git-protocol https \
  --web



! First copy your one-time code: 4AFD-66C1
Open this URL to continue in your web browser: https://github.com/login/device
✓ Authentication complete.
- gh config set -h github.com git_protocol https
✓ Configured git protocol
! Authentication credentials saved in plain text
✓ Logged in as Rohil121


# Bharat Portfolio Lab v0.6

## Machine Learning and Trading Research

### Objective

Test whether machine-learning signals can improve the risk-adjusted performance of an Indian equity portfolio after transaction costs, turnover and strict out-of-sample validation.

The initial research universe will be the India 10 portfolio, while the final production modules will support arbitrary eligible Indian listed equities.

## Fixed v0.6 Scope

### Prediction tasks

1. Predict each stock’s forward 21-trading-day return.
2. Estimate the probability of a positive forward return.
3. Estimate the probability of outperforming the Nifty 50 over the same period.

### Candidate features

- 1-month, 3-month, 6-month and 12-month momentum
- 21-day and 63-day realised volatility
- Recent drawdown and distance from rolling high
- Moving-average trend indicators
- Relative strength versus the Nifty 50
- Rolling beta and benchmark correlation
- Nifty 50 trend, volatility and market regime
- Lagged stock returns
- Volume-based indicators where reliable

### Candidate models

- Historical-mean baseline
- Momentum baseline
- Linear regression
- Ridge and Lasso regression
- Logistic regression
- Random forest
- Gradient boosting

### Trading strategies

- Top-ranked long-only portfolio
- Probability-weighted portfolio
- ML signal with inverse-volatility sizing
- Regime-aware ML portfolio
- Equal-weight and India 10 benchmarks

### Evaluation principles

- Expanding-window walk-forward validation
- No random train-test split
- No future information in model features
- One-day signal execution lag
- Monthly rebalancing
- Transaction costs and turnover
- CAGR, volatility, Sharpe ratio and maximum drawdown
- Hit rate, prediction error and rank information coefficient
- Performance comparison across market regimes

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


# Project folders
REPO_ROOT = Path("/content/bharat-portfolio-lab")

PROCESSED_DATA_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "ml_trading"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "ml_trading"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Research assumptions
RANDOM_SEED = 42
TRADING_DAYS_PER_YEAR = 252
FORWARD_HORIZON_DAYS = 21
MINIMUM_TRAINING_DAYS = 756

RISK_FREE_RATE = 0.065
ONE_WAY_TRANSACTION_COST = 0.0015

BENCHMARK_TICKER = "^NSEI"
BENCHMARK_NAME = "Nifty 50"


# India 10 research universe
INDIA_10_TICKERS = [
    "HDFCBANK.NS",
    "TCS.NS",
    "HINDUNILVR.NS",
    "SUNPHARMA.NS",
    "POWERGRID.NS",
    "BHARTIARTL.NS",
    "LT.NS",
    "M&M.NS",
    "BEL.NS",
    "TRENT.NS",
]

np.random.seed(
    RANDOM_SEED
)

print("v0.6 research configuration initialised.")
print("Research universe:", len(INDIA_10_TICKERS), "stocks")
print("Prediction horizon:", FORWARD_HORIZON_DAYS, "trading days")
print("Minimum training history:", MINIMUM_TRAINING_DAYS, "trading days")
print("Benchmark:", BENCHMARK_NAME)
print("Transaction cost:", f"{ONE_WAY_TRANSACTION_COST:.2%}")

v0.6 research configuration initialised.
Research universe: 10 stocks
Prediction horizon: 21 trading days
Minimum training history: 756 trading days
Benchmark: Nifty 50
Transaction cost: 0.15%


## Market Data Collection

Download daily adjusted prices and trading volumes for the India 10 stocks and the Nifty 50 benchmark.

The dataset begins in 2015 to provide enough history for:

- 12-month momentum features
- Three-year minimum training windows
- Expanding-window validation
- Multiple market regimes
- Strict out-of-sample testing

In [3]:
import yfinance as yf

DATA_START_DATE = "2015-01-01"
DATA_END_DATE = "2026-07-31"

ALL_TICKERS = (
    INDIA_10_TICKERS
    + [BENCHMARK_TICKER]
)

raw_market_data = yf.download(
    tickers=ALL_TICKERS,
    start=DATA_START_DATE,
    end=DATA_END_DATE,
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True,
)

if raw_market_data.empty:
    raise RuntimeError(
        "No market data were downloaded."
    )

if not isinstance(
    raw_market_data.columns,
    pd.MultiIndex,
):
    raise RuntimeError(
        "Unexpected yFinance column format."
    )


# Extract adjusted close prices and volumes
close_prices = (
    raw_market_data["Close"]
    .reindex(
        columns=ALL_TICKERS
    )
    .sort_index()
)

trading_volume = (
    raw_market_data["Volume"]
    .reindex(
        columns=ALL_TICKERS
    )
    .sort_index()
)


# Remove dates on which every security is missing
close_prices = close_prices.dropna(
    how="all"
)

trading_volume = trading_volume.reindex(
    close_prices.index
)


# Create a data-quality summary
quality_summary = pd.DataFrame(
    {
        "First Valid Date": (
            close_prices.apply(
                lambda series: series.first_valid_index()
            )
        ),
        "Last Valid Date": (
            close_prices.apply(
                lambda series: series.last_valid_index()
            )
        ),
        "Price Observations": (
            close_prices.notna().sum()
        ),
        "Missing Prices": (
            close_prices.isna().sum()
        ),
        "Volume Observations": (
            trading_volume.notna().sum()
        ),
    }
)

quality_summary[
    "Price Coverage"
] = (
    quality_summary[
        "Price Observations"
    ]
    / len(close_prices)
)

quality_summary[
    "Sufficient for ML"
] = (
    quality_summary[
        "Price Observations"
    ]
    >= (
        MINIMUM_TRAINING_DAYS
        + 252
    )
)


print("ML MARKET DATA CHECK")
print("=" * 65)
print(
    "Downloaded period:",
    close_prices.index.min().date(),
    "to",
    close_prices.index.max().date(),
)
print(
    "Trading dates:",
    len(close_prices),
)
print(
    "India 10 stocks:",
    len(INDIA_10_TICKERS),
)
print(
    "Benchmark included:",
    BENCHMARK_TICKER in close_prices.columns,
)
print(
    "All securities sufficient for ML:",
    quality_summary[
        "Sufficient for ML"
    ].all(),
)

display(
    quality_summary
)

ML MARKET DATA CHECK
Downloaded period: 2015-01-01 to 2026-07-30
Trading dates: 2861
India 10 stocks: 10
Benchmark included: True
All securities sufficient for ML: True


,First Valid Date,Last Valid Date,Price Observations,Missing Prices,Volume Observations,Price Coverage,Sufficient for ML
Ticker,,,,,,,
HDFCBANK.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
TCS.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
HINDUNILVR.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
SUNPHARMA.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
POWERGRID.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
BHARTIARTL.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
LT.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
M&M.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
BEL.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True


## Leakage-Safe Feature and Target Dataset

Each observation represents one stock on one trading date.

Features use only information available on or before that date. The prediction targets measure:

- Forward 21-trading-day stock return
- Whether the forward return is positive
- Forward excess return versus the Nifty 50
- Whether the stock outperforms the Nifty 50

The final trading signal will be executed with a one-day lag during backtesting.

In [4]:
# Daily return series
stock_daily_returns = (
    close_prices[
        INDIA_10_TICKERS
    ]
    .pct_change(
        fill_method=None
    )
)

benchmark_price = (
    close_prices[
        BENCHMARK_TICKER
    ]
)

benchmark_daily_return = (
    benchmark_price
    .pct_change(
        fill_method=None
    )
)


# Benchmark-wide features repeated for every stock
benchmark_features = pd.DataFrame(
    index=close_prices.index
)

benchmark_features[
    "benchmark_return_1d"
] = benchmark_daily_return

benchmark_features[
    "benchmark_momentum_21d"
] = (
    benchmark_price
    .pct_change(
        21,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_momentum_63d"
] = (
    benchmark_price
    .pct_change(
        63,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_momentum_126d"
] = (
    benchmark_price
    .pct_change(
        126,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_volatility_21d"
] = (
    benchmark_daily_return
    .rolling(
        21
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

benchmark_features[
    "benchmark_volatility_63d"
] = (
    benchmark_daily_return
    .rolling(
        63
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

benchmark_features[
    "benchmark_drawdown_252d"
] = (
    benchmark_price
    / benchmark_price
    .rolling(
        252
    )
    .max()
    - 1
)

benchmark_features[
    "benchmark_ma_gap_200d"
] = (
    benchmark_price
    / benchmark_price
    .rolling(
        200
    )
    .mean()
    - 1
)

benchmark_features[
    "benchmark_bull_regime"
] = (
    benchmark_price
    > benchmark_price
    .rolling(
        200
    )
    .mean()
).astype(float)


# Forward benchmark target
benchmark_forward_return_21d = (
    benchmark_price.shift(
        -FORWARD_HORIZON_DAYS
    )
    / benchmark_price
    - 1
)


# Build one feature frame per stock
stock_feature_frames = {}

for ticker in INDIA_10_TICKERS:

    stock_price = (
        close_prices[
            ticker
        ]
    )

    stock_return = (
        stock_daily_returns[
            ticker
        ]
    )

    stock_volume = (
        trading_volume[
            ticker
        ]
    )

    stock_frame = pd.DataFrame(
        index=close_prices.index
    )

    # Recent returns and momentum
    stock_frame[
        "return_1d"
    ] = stock_return

    stock_frame[
        "return_5d"
    ] = (
        stock_price
        .pct_change(
            5,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_21d"
    ] = (
        stock_price
        .pct_change(
            21,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_63d"
    ] = (
        stock_price
        .pct_change(
            63,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_126d"
    ] = (
        stock_price
        .pct_change(
            126,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_252d"
    ] = (
        stock_price
        .pct_change(
            252,
            fill_method=None
        )
    )

    # Risk and drawdown
    stock_frame[
        "volatility_21d"
    ] = (
        stock_return
        .rolling(
            21
        )
        .std()
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    stock_frame[
        "volatility_63d"
    ] = (
        stock_return
        .rolling(
            63
        )
        .std()
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    stock_frame[
        "drawdown_252d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            252
        )
        .max()
        - 1
    )

    # Trend indicators
    stock_frame[
        "ma_gap_21d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            21
        )
        .mean()
        - 1
    )

    stock_frame[
        "ma_gap_63d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            63
        )
        .mean()
        - 1
    )

    stock_frame[
        "ma_gap_200d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            200
        )
        .mean()
        - 1
    )

    # Relative strength versus Nifty 50
    stock_frame[
        "relative_strength_63d"
    ] = (
        stock_frame[
            "momentum_63d"
        ]
        - benchmark_features[
            "benchmark_momentum_63d"
        ]
    )

    stock_frame[
        "relative_strength_126d"
    ] = (
        stock_frame[
            "momentum_126d"
        ]
        - benchmark_features[
            "benchmark_momentum_126d"
        ]
    )

    # Rolling market sensitivity
    rolling_covariance = (
        stock_return
        .rolling(
            63
        )
        .cov(
            benchmark_daily_return
        )
    )

    rolling_benchmark_variance = (
        benchmark_daily_return
        .rolling(
            63
        )
        .var()
    )

    stock_frame[
        "beta_63d"
    ] = (
        rolling_covariance
        / rolling_benchmark_variance
    )

    stock_frame[
        "benchmark_correlation_63d"
    ] = (
        stock_return
        .rolling(
            63
        )
        .corr(
            benchmark_daily_return
        )
    )

    # Volume features
    stock_frame[
        "volume_ratio_21d"
    ] = (
        stock_volume
        / stock_volume
        .rolling(
            21
        )
        .mean()
        - 1
    )

    stock_frame[
        "volume_trend_21_63d"
    ] = (
        stock_volume
        .rolling(
            21
        )
        .mean()
        / stock_volume
        .rolling(
            63
        )
        .mean()
        - 1
    )

    # Add market-wide features
    stock_frame = stock_frame.join(
        benchmark_features
    )

    # Forward targets
    forward_return = (
        stock_price.shift(
            -FORWARD_HORIZON_DAYS
        )
        / stock_price
        - 1
    )

    forward_excess_return = (
        forward_return
        - benchmark_forward_return_21d
    )

    stock_frame[
        "forward_return_21d"
    ] = forward_return

    stock_frame[
        "forward_excess_return_21d"
    ] = forward_excess_return

    stock_frame[
        "positive_return_target"
    ] = (
        forward_return
        .gt(0)
        .where(
            forward_return.notna()
        )
        .astype(float)
    )

    stock_frame[
        "outperform_target"
    ] = (
        forward_excess_return
        .gt(0)
        .where(
            forward_excess_return.notna()
        )
        .astype(float)
    )

    stock_feature_frames[
        ticker
    ] = stock_frame


# Convert into one stock-date panel
ml_panel_raw = (
    pd.concat(
        stock_feature_frames,
        names=[
            "Ticker",
            "Date",
        ],
    )
    .reset_index()
)


feature_columns = [
    "return_1d",
    "return_5d",
    "momentum_21d",
    "momentum_63d",
    "momentum_126d",
    "momentum_252d",
    "volatility_21d",
    "volatility_63d",
    "drawdown_252d",
    "ma_gap_21d",
    "ma_gap_63d",
    "ma_gap_200d",
    "relative_strength_63d",
    "relative_strength_126d",
    "beta_63d",
    "benchmark_correlation_63d",
    "volume_ratio_21d",
    "volume_trend_21_63d",
    "benchmark_return_1d",
    "benchmark_momentum_21d",
    "benchmark_momentum_63d",
    "benchmark_momentum_126d",
    "benchmark_volatility_21d",
    "benchmark_volatility_63d",
    "benchmark_drawdown_252d",
    "benchmark_ma_gap_200d",
    "benchmark_bull_regime",
]

target_columns = [
    "forward_return_21d",
    "forward_excess_return_21d",
    "positive_return_target",
    "outperform_target",
]


# Retain only observations with complete features and targets
ml_panel = (
    ml_panel_raw
    .dropna(
        subset=(
            feature_columns
            + target_columns
        )
    )
    .sort_values(
        [
            "Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# Store classification targets as integers
ml_panel[
    "positive_return_target"
] = (
    ml_panel[
        "positive_return_target"
    ]
    .astype(int)
)

ml_panel[
    "outperform_target"
] = (
    ml_panel[
        "outperform_target"
    ]
    .astype(int)
)


# Save a reproducible research dataset
feature_dataset_path = (
    PROCESSED_DATA_DIR
    / "india10_ml_feature_panel.csv"
)

ml_panel.to_csv(
    feature_dataset_path,
    index=False,
)


# Validation
assert ml_panel[
    feature_columns
].notna().all().all()

assert ml_panel[
    target_columns
].notna().all().all()

assert set(
    ml_panel[
        "Ticker"
    ].unique()
) == set(
    INDIA_10_TICKERS
)

assert (
    ml_panel[
        "Date"
    ].max()
    <= close_prices.index[
        -FORWARD_HORIZON_DAYS - 1
    ]
)


print("ML FEATURE DATASET CHECK")
print("=" * 65)
print(
    "Panel observations:",
    f"{len(ml_panel):,}",
)
print(
    "Stocks:",
    ml_panel[
        "Ticker"
    ].nunique(),
)
print(
    "Features:",
    len(feature_columns),
)
print(
    "Dataset period:",
    ml_panel[
        "Date"
    ].min().date(),
    "to",
    ml_panel[
        "Date"
    ].max().date(),
)
print(
    "Positive-return rate:",
    f"{ml_panel['positive_return_target'].mean():.2%}",
)
print(
    "Nifty outperformance rate:",
    f"{ml_panel['outperform_target'].mean():.2%}",
)
print(
    "Missing feature values:",
    int(
        ml_panel[
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
)
print(
    "Saved dataset:",
    feature_dataset_path,
)

display(
    ml_panel.head()
)

ML FEATURE DATASET CHECK
Panel observations: 11,260
Stocks: 10
Features: 27
Dataset period: 2017-08-22 to 2026-01-14
Positive-return rate: 60.67%
Nifty outperformance rate: 54.83%
Missing feature values: 0
Saved dataset: /content/bharat-portfolio-lab/data/processed/ml_trading/india10_ml_feature_panel.csv


,Ticker,Date,return_1d,return_5d,momentum_21d,momentum_63d,momentum_126d,momentum_252d,volatility_21d,volatility_63d,...,benchmark_momentum_126d,benchmark_volatility_21d,benchmark_volatility_63d,benchmark_drawdown_252d,benchmark_ma_gap_200d,benchmark_bull_regime,forward_return_21d,forward_excess_return_21d,positive_return_target,outperform_target
0,BEL.NS,2017-08-22,-0.005025,0.011064,0.045481,0.042139,0.181372,0.487748,0.299816,0.248630,...,0.112503,0.100589,0.086879,-0.034514,0.080253,1.0,0.056117,0.019626,1,1
1,BHARTIARTL.NS,2017-08-22,0.007038,0.033166,0.026258,0.137848,0.157139,0.216692,0.195072,0.205047,...,0.112503,0.100589,0.086879,-0.034514,0.080253,1.0,-0.052121,-0.088611,0,0
2,HDFCBANK.NS,2017-08-22,0.002180,-0.005805,0.025807,0.120775,0.324353,0.431276,0.143402,0.121879,...,0.112503,0.100589,0.086879,-0.034514,0.080253,1.0,0.052719,0.016228,1,1
3,HINDUNILVR.NS,2017-08-22,0.006841,0.037986,0.035343,0.181083,0.430533,0.305442,0.190381,0.176395,...,0.112503,0.100589,0.086879,-0.034514,0.080253,1.0,0.047599,0.011109,1,1
4,LT.NS,2017-08-22,-0.008340,-0.025460,-0.040592,-0.013178,0.168379,0.165582,0.158313,0.197533,...,0.112503,0.100589,0.086879,-0.034514,0.080253,1.0,0.098730,0.062239,1,1
